In [1]:
"""
Student Submission and Validation System for CNN Competition

This script allows students to:
1. Test their submission locally with their own test set
2. Validate that all required files are present and correct
3. Automatically submit to a private folder only visible to them and instructors

Setup instructions for instructors:
1. Create a Google Drive folder for submissions
2. Create individual subfolders for each group (Group_A, Group_B, etc.)
3. Share each subfolder with only that group and instructors
4. Give students the path to their specific folder
"""

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms, models
from PIL import Image
import os
import shutil
import json
from datetime import datetime
import importlib.util
import sys
import hashlib

# Mount Google Drive (if using Colab)
try:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE_PATH = '/content/drive/MyDrive/BIP Project Team L/CNN Challenge'
except:
    BASE_PATH = './'

class Config:
    # STUDENT: Set these paths
    # Path to YOUR validation test set (your own images for testing)
    student_test_dir = os.path.join(BASE_PATH, 'my_validation_set')

    # Path to YOUR submission folder (where you have model.pth, info.json, etc.)
    student_submission_dir = os.path.join(BASE_PATH, 'my_submission')


    # INSTRUCTOR WILL PROVIDE: Path to your private submission folder in Drive
    # This folder is shared ONLY with you and instructors
    # Example: '/content/drive/MyDrive/CNN_Competition_Submissions/Group_A/'
    submission_target_dir = os.path.join(BASE_PATH, 'CNN_Competition_Submissions/Group L/')

    # Parameters
    img_size = 224
    batch_size = 32
    num_classes = 2
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')


class GlaucomaDataset(Dataset):
    """Simple dataset for validation"""
    def __init__(self, root_dir, transform=None):
        self.root_dir = root_dir
        self.transform = transform
        self.images = []
        self.labels = []

        for label, class_name in enumerate(['glaucoma', 'normal']):
            class_dir = os.path.join(root_dir, class_name)
            if os.path.exists(class_dir):
                for img_name in os.listdir(class_dir):
                    if img_name.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp')):
                        self.images.append(os.path.join(class_dir, img_name))
                        self.labels.append(label)

        print(f"  Found {len(self.images)} images")
        if len(self.labels) > 0:
            print(f"  Normal: {self.labels.count(0)}, Glaucoma: {self.labels.count(1)}")

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        img_path = self.images[idx]
        image = Image.open(img_path).convert('RGB')
        label = self.labels[idx]

        if self.transform:
            image = self.transform(image)

        return image, label


def get_default_test_transform():
    """Default transforms"""
    return transforms.Compose([
        transforms.Resize((Config.img_size, Config.img_size)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])


def load_student_transforms(transform_file_path):
    """Load custom transforms from student"""
    if not os.path.exists(transform_file_path):
        return None

    try:
        spec = importlib.util.spec_from_file_location("student_transforms", transform_file_path)
        student_transforms_module = importlib.util.module_from_spec(spec)
        sys.modules["student_transforms"] = student_transforms_module
        spec.loader.exec_module(student_transforms_module)

        if hasattr(student_transforms_module, 'get_test_transform'):
            return student_transforms_module.get_test_transform()
    except Exception as e:
        print(f"  ⚠ Error loading transforms: {str(e)}")

    return None


def load_student_model_architecture(model_file_path):
    """Load custom model architecture"""
    if not os.path.exists(model_file_path):
        return None

    try:
        spec = importlib.util.spec_from_file_location("student_model", model_file_path)
        student_model_module = importlib.util.module_from_spec(spec)
        sys.modules["student_model"] = student_model_module
        spec.loader.exec_module(student_model_module)

        if hasattr(student_model_module, 'get_model'):
            return student_model_module.get_model
    except Exception as e:
        print(f"  ✗ Error loading model architecture: {str(e)}")

    return None


def validate_submission_structure(submission_dir):
    """
    Validate that submission has all required files and correct structure

    Returns:
        (bool, list): (is_valid, list_of_issues)
    """
    print("\n" + "="*80)
    print("STEP 1: VALIDATING SUBMISSION STRUCTURE")
    print("="*80 + "\n")

    issues = []

    # Check directory exists
    if not os.path.exists(submission_dir):
        issues.append(f"Submission directory not found: {submission_dir}")
        return False, issues

    print(f"✓ Submission directory found: {submission_dir}\n")

    # Required files
    required_files = {
        'model.pth': 'Trained model weights',
        'info.json': 'Model information',
        'model.py': 'Model architecture definition'
    }

    optional_files = {
        'transforms.py': 'Custom transforms (optional)',
    }

    print("Checking required files:")
    for filename, description in required_files.items():
        filepath = os.path.join(submission_dir, filename)
        if os.path.exists(filepath):
            file_size = os.path.getsize(filepath)
            print(f"  ✓ {filename:25s} - {description} ({file_size:,} bytes)")
        else:
            print(f"  ✗ {filename:25s} - MISSING")
            issues.append(f"Required file missing: {filename}")

    print("\nChecking optional files:")
    for filename, description in optional_files.items():
        filepath = os.path.join(submission_dir, filename)
        if os.path.exists(filepath):
            file_size = os.path.getsize(filepath)
            print(f"  ✓ {filename:25s} - {description} ({file_size:,} bytes)")
        else:
            print(f"  - {filename:25s} - Not provided (optional)")

    # Validate info.json content
    info_path = os.path.join(submission_dir, 'info.json')
    if os.path.exists(info_path):
        print("\nValidating info.json content:")
        try:
            with open(info_path, 'r') as f:
                info = json.load(f)

            required_fields = ['group_id']
            for field in required_fields:
                if field in info:
                    print(f"  ✓ {field}: {info[field]}")
                else:
                    print(f"  ✗ {field}: MISSING")
                    issues.append(f"Required field '{field}' missing in info.json")

            # Optional fields
            optional_fields = ['architecture', 'description', 'training_details']
            for field in optional_fields:
                if field in info:
                    print(f"  ✓ {field}: Present")

        except json.JSONDecodeError as e:
            issues.append(f"info.json is not valid JSON: {str(e)}")
            print(f"  ✗ Invalid JSON format")

    # Validate model.py
    model_py_path = os.path.join(submission_dir, 'model.py')
    if os.path.exists(model_py_path):
        print("\nValidating model.py:")
        model_fn = load_student_model_architecture(model_py_path)
        if model_fn is not None:
            print("  ✓ get_model() function found")
            try:
                # Try to instantiate the model
                test_model = model_fn(num_classes=2)
                print("  ✓ Model can be instantiated")
            except Exception as e:
                issues.append(f"Error creating model from model.py: {str(e)}")
                print(f"  ✗ Error creating model: {str(e)}")
        else:
            issues.append("model.py doesn't contain get_model() function")
            print("  ✗ get_model() function not found")

    # Summary
    print("\n" + "="*80)
    if len(issues) == 0:
        print("✓ VALIDATION PASSED - All checks successful!")
    else:
        print("✗ VALIDATION FAILED - Issues found:")
        for i, issue in enumerate(issues, 1):
            print(f"  {i}. {issue}")
    print("="*80)

    return len(issues) == 0, issues


def test_model_loading(submission_dir):
    """
    Test that the model can be loaded successfully

    Returns:
        (bool, model or None): (success, loaded_model)
    """
    print("\n" + "="*80)
    print("STEP 2: TESTING MODEL LOADING")
    print("="*80 + "\n")

    model_path = os.path.join(submission_dir, 'model.pth')
    model_py_path = os.path.join(submission_dir, 'model.py')

    try:
        # Load architecture
        print("Loading model architecture from model.py...")
        model_fn = load_student_model_architecture(model_py_path)

        if model_fn is None:
            print("✗ Failed to load model architecture")
            return False, None

        model = model_fn(num_classes=Config.num_classes)
        print("✓ Model architecture loaded")

        # Load weights
        print("\nLoading model weights from model.pth...")
        state_dict = torch.load(model_path, map_location=Config.device, weights_only=True)
        model.load_state_dict(state_dict)
        print("✓ Model weights loaded")

        # Move to device
        model = model.to(Config.device)
        model.eval()
        print(f"✓ Model moved to {Config.device}")

        # Test with dummy input
        print("\nTesting model with dummy input...")
        dummy_input = torch.randn(1, 3, Config.img_size, Config.img_size).to(Config.device)
        with torch.no_grad():
            output = model(dummy_input)

        if output.shape[1] == Config.num_classes:
            print(f"✓ Model output shape correct: {output.shape}")
        else:
            print(f"✗ Model output shape incorrect: {output.shape}, expected (1, {Config.num_classes})")
            return False, None

        print("\n" + "="*80)
        print("✓ MODEL LOADING TEST PASSED")
        print("="*80)

        return True, model

    except Exception as e:
        print(f"\n✗ Error loading model: {str(e)}")
        print("\n" + "="*80)
        print("✗ MODEL LOADING TEST FAILED")
        print("="*80)
        return False, None


def run_validation_test(submission_dir, test_dir, model):
    """
    Run a quick validation on student's own test set

    Returns:
        (bool, dict): (success, results)
    """
    print("\n" + "="*80)
    print("STEP 3: RUNNING VALIDATION TEST")
    print("="*80 + "\n")

    if not os.path.exists(test_dir):
        print(f"⚠ Test directory not found: {test_dir}")
        print("  You can skip this step, but it's recommended to test locally first")
        return True, None

    try:
        # Load transforms
        transforms_path = os.path.join(submission_dir, 'transforms.py')
        test_transform = load_student_transforms(transforms_path)
        if test_transform is None:
            test_transform = get_default_test_transform()
            print("Using default transforms")
        else:
            print("Using custom transforms from transforms.py")

        # Load dataset
        print(f"\nLoading test images from: {test_dir}")
        test_dataset = GlaucomaDataset(test_dir, transform=test_transform)

        if len(test_dataset) == 0:
            print("⚠ No images found in test directory")
            return True, None

        test_loader = DataLoader(test_dataset, batch_size=Config.batch_size,
                                shuffle=False, num_workers=0)

        # Run inference
        print("\nRunning inference...")
        model.eval()
        all_predictions = []
        all_labels = []

        with torch.no_grad():
            for images, labels in test_loader:
                images = images.to(Config.device)
                outputs = model(images)
                _, predicted = torch.max(outputs, 1)

                all_predictions.extend(predicted.cpu().numpy())
                all_labels.extend(labels.numpy())

        # Calculate simple accuracy
        correct = sum([p == l for p, l in zip(all_predictions, all_labels)])
        accuracy = correct / len(all_labels)

        print(f"\nValidation Results:")
        print(f"  Total images: {len(all_labels)}")
        print(f"  Correct: {correct}")
        print(f"  Accuracy: {accuracy:.2%}")

        print("\n" + "="*80)
        print("✓ VALIDATION TEST COMPLETED")
        print("="*80)

        return True, {'accuracy': accuracy, 'total': len(all_labels), 'correct': correct}

    except Exception as e:
        print(f"\n✗ Error during validation: {str(e)}")
        print("\n" + "="*80)
        print("✗ VALIDATION TEST FAILED")
        print("="*80)
        return False, None


def generate_submission_report(submission_dir, validation_results):
    """Generate a report about the submission"""
    report_path = os.path.join(submission_dir, 'submission_report.txt')

    with open(report_path, 'w') as f:
        f.write("="*80 + "\n")
        f.write("SUBMISSION VALIDATION REPORT\n")
        f.write("="*80 + "\n\n")
        f.write(f"Timestamp: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n\n")

        # Load info
        info_path = os.path.join(submission_dir, 'info.json')
        if os.path.exists(info_path):
            with open(info_path, 'r') as info_f:
                info = json.load(info_f)
                f.write("Submission Info:\n")
                f.write(f"  Group ID: {info.get('group_id', 'N/A')}\n")
                f.write(f"  Architecture: {info.get('architecture', 'N/A')}\n")
                if 'description' in info:
                    f.write(f"  Description: {info['description']}\n")
                f.write("\n")

        # Files included
        f.write("Files Included:\n")
        for filename in os.listdir(submission_dir):
            if not filename.startswith('.'):
                filepath = os.path.join(submission_dir, filename)
                if os.path.isfile(filepath):
                    size = os.path.getsize(filepath)
                    f.write(f"  - {filename} ({size:,} bytes)\n")
        f.write("\n")

        # Validation results
        if validation_results:
            f.write("Local Validation Results:\n")
            f.write(f"  Accuracy: {validation_results['accuracy']:.2%}\n")
            f.write(f"  Total images: {validation_results['total']}\n")
            f.write(f"  Correct predictions: {validation_results['correct']}\n")
            f.write("\n")

        f.write("="*80 + "\n")
        f.write("Validation completed successfully!\n")
        f.write("Ready for official evaluation.\n")
        f.write("="*80 + "\n")

    return report_path


def submit_to_instructor_folder(submission_dir, target_dir):
    """
    Copy submission to instructor-accessible folder

    This folder should be:
    - Shared with the specific student group
    - Shared with instructors
    - NOT visible to other students
    """
    print("\n" + "="*80)
    print("STEP 4: SUBMITTING TO INSTRUCTOR FOLDER")
    print("="*80 + "\n")

    # Check if target directory exists and is accessible
    if not os.path.exists(target_dir):
        print(f"⚠ Target directory not found: {target_dir}")
        print("\nPlease make sure:")
        print("1. You're connected to Google Drive")
        print("2. The folder path is correct")
        print("3. The folder has been shared with you by instructors")
        return False

    # Create timestamped submission folder
    timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
    submission_folder_name = f"submission_{timestamp}"
    final_target = os.path.join(target_dir, submission_folder_name)

    try:
        print(f"Creating submission folder: {submission_folder_name}")
        os.makedirs(final_target, exist_ok=True)

        # Files to copy
        files_to_copy = ['model.pth', 'info.json', 'model.py', 'transforms.py',
                        'preprocessing_config.json', 'submission_report.txt']

        print("\nCopying files:")
        copied_files = []
        for filename in files_to_copy:
            src = os.path.join(submission_dir, filename)
            if os.path.exists(src):
                dst = os.path.join(final_target, filename)
                shutil.copy2(src, dst)
                file_size = os.path.getsize(dst)
                print(f"  ✓ {filename} ({file_size:,} bytes)")
                copied_files.append(filename)
            elif filename in ['model.pth', 'info.json', 'model.py']:
                print(f"  ✗ {filename} - REQUIRED FILE MISSING!")
                return False

        # Create submission metadata
        metadata = {
            'submission_timestamp': timestamp,
            'files_included': copied_files,
            'submission_complete': True
        }

        metadata_path = os.path.join(final_target, 'submission_metadata.json')
        with open(metadata_path, 'w') as f:
            json.dump(metadata, f, indent=4)

        print(f"\n✓ Submission copied to: {final_target}")
        print("\n" + "="*80)
        print("✓ SUBMISSION SUCCESSFUL!")
        print("="*80)
        print("\nYour submission has been saved to a folder that only you and")
        print("the instructors can access. Other students cannot see your files.")
        print(f"\nSubmission ID: {timestamp}")

        return True

    except Exception as e:
        print(f"\n✗ Error during submission: {str(e)}")
        print("\nPossible issues:")
        print("- No write permission to target folder")
        print("- Target folder not properly shared")
        print("- Network/Drive connection issues")
        return False


def main():
    """
    Main function to validate and submit student work
    """
    print("\n" + "="*80)
    print("CNN COMPETITION - STUDENT VALIDATION AND SUBMISSION SYSTEM")
    print("="*80)
    print("\nThis script will:")
    print("1. Validate your submission structure")
    print("2. Test that your model loads correctly")
    print("3. Run validation on your test set (optional)")
    print("4. Submit to your private folder (visible only to you and instructors)")
    print("\n" + "="*80 + "\n")

    # Configuration check
    print("Checking configuration...")
    print(f"  Submission directory: {Config.student_submission_dir}")
    print(f"  Test directory: {Config.student_test_dir}")
    print(f"  Target directory: {Config.submission_target_dir}")
    print(f"  Device: {Config.device}")

    # Ask for confirmation
    print("\n" + "="*80)
    response = input("Do you want to proceed? (yes/no): ").strip().lower()
    if response not in ['yes', 'y']:
        print("Validation cancelled.")
        return

    # Step 1: Validate structure
    is_valid, issues = validate_submission_structure(Config.student_submission_dir)
    if not is_valid:
        print("\n❌ VALIDATION FAILED")
        print("Please fix the issues above before submitting.")
        return

    # Step 2: Test model loading
    success, model = test_model_loading(Config.student_submission_dir)
    if not success:
        print("\n❌ MODEL LOADING FAILED")
        print("Please fix your model.py or model.pth file.")
        return

    # Step 3: Run validation (optional)
    print("\n" + "="*80)
    if os.path.exists(Config.student_test_dir):
        response = input("Run validation on your test set? (yes/no): ").strip().lower()
        if response in ['yes', 'y']:
            success, val_results = run_validation_test(
                Config.student_submission_dir,
                Config.student_test_dir,
                model
            )
            if not success:
                print("\n⚠ Validation test had issues, but you can still submit.")
        else:
            val_results = None
    else:
        print(f"Test directory not found: {Config.student_test_dir}")
        print("Skipping validation test (this is optional)")
        val_results = None

    # Generate report
    report_path = generate_submission_report(Config.student_submission_dir, val_results)
    print(f"\n✓ Validation report generated: {report_path}")

    # Step 4: Submit
    print("\n" + "="*80)
    print("Ready to submit to instructor folder!")
    print("This will copy your files to a private folder shared only with")
    print("you and the instructors. Other students will NOT be able to see it.")
    response = input("\nProceed with submission? (yes/no): ").strip().lower()

    if response not in ['yes', 'y']:
        print("\nSubmission cancelled.")
        print("Your files have been validated but not submitted.")
        print("You can run this script again when ready to submit.")
        return

    # Submit
    success = submit_to_instructor_folder(
        Config.student_submission_dir,
        Config.submission_target_dir
    )

    if success:
        print("\n" + "="*80)
        print("🎉 ALL DONE!")
        print("="*80)
        print("\nYour submission has been successfully validated and submitted.")
        print("The instructors will evaluate your model on the official test set.")
        print("\nYou will receive your results once all evaluations are complete.")
    else:
        print("\n❌ SUBMISSION FAILED")
        print("Please check the errors above and try again.")
        print("Contact instructors if you continue having issues.")


if __name__ == '__main__':
    main()

Mounted at /content/drive

CNN COMPETITION - STUDENT VALIDATION AND SUBMISSION SYSTEM

This script will:
1. Validate your submission structure
2. Test that your model loads correctly
3. Run validation on your test set (optional)
4. Submit to your private folder (visible only to you and instructors)


Checking configuration...
  Submission directory: /content/drive/MyDrive/BIP Project Team L/CNN Challenge/my_submission
  Test directory: /content/drive/MyDrive/BIP Project Team L/CNN Challenge/my_validation_set
  Target directory: /content/drive/MyDrive/BIP Project Team L/CNN Challenge/CNN_Competition_Submissions/Group L/
  Device: cuda

Do you want to proceed? (yes/no): yes

STEP 1: VALIDATING SUBMISSION STRUCTURE

✓ Submission directory found: /content/drive/MyDrive/BIP Project Team L/CNN Challenge/my_submission

Checking required files:
  ✓ model.pth                 - Trained model weights (343,262,941 bytes)
  ✓ info.json                 - Model information (865 bytes)
  ✓ model.py   

/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)


  ✓ Model can be instantiated

✓ VALIDATION PASSED - All checks successful!

STEP 2: TESTING MODEL LOADING

Loading model architecture from model.py...
✓ Model architecture loaded

Loading model weights from model.pth...

✗ Error loading model: Error(s) in loading state_dict for ResNet:
	Missing key(s) in state_dict: "conv1.weight", "bn1.weight", "bn1.bias", "bn1.running_mean", "bn1.running_var", "layer1.0.conv1.weight", "layer1.0.bn1.weight", "layer1.0.bn1.bias", "layer1.0.bn1.running_mean", "layer1.0.bn1.running_var", "layer1.0.conv2.weight", "layer1.0.bn2.weight", "layer1.0.bn2.bias", "layer1.0.bn2.running_mean", "layer1.0.bn2.running_var", "layer1.0.conv3.weight", "layer1.0.bn3.weight", "layer1.0.bn3.bias", "layer1.0.bn3.running_mean", "layer1.0.bn3.running_var", "layer1.0.downsample.0.weight", "layer1.0.downsample.1.weight", "layer1.0.downsample.1.bias", "layer1.0.downsample.1.running_mean", "layer1.0.downsample.1.running_var", "layer1.1.conv1.weight", "layer1.1.bn1.weight", "laye